# 深度错误归因 ：弱监督规则的系统性破绽

## 1. 分析背景
在独立测试集上，本系统微调的 BERT 模型虽然取得了高达 91.65% 的整体准确率，且极少发生“越级误判”（如 0 判 2），但误差仍集中在相邻的灰色地带（特别是真实标签为 0，模型预测为 1 的情况）。

本 Notebook 旨在提取这些边界模糊的 Bad Case 进行逐条语义剖析。

# 我们的目标不是证明大模型“变笨了”，而是利用大模型的预测偏差，反向审查我们初期设计的“弱监督特征加权公式”是否存在先验的逻辑漏洞。


In [3]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. 路径配置
test_path = '../data/processed/test_dataset.csv'
final_model_dir = '../models/final_bert_qa_model'

# 2. 加载模型与数据
print("正在加载模型与测试集...")
tokenizer = AutoTokenizer.from_pretrained(final_model_dir)
model = AutoModelForSequenceClassification.from_pretrained(final_model_dir)

# 抽取 2000 条测试数据来寻找 Bad Case
df_test = pd.read_csv(test_path).head(2000)
df_clean = df_test[['model_input', 'final_label']].rename(columns={'final_label': 'label'})
test_dataset = Dataset.from_pandas(df_clean)

def tokenize_function(examples):
    return tokenizer(examples["model_input"], padding="max_length", truncation=True, max_length=256)

print("正在处理文本张量化...")
tokenized_test = test_dataset.map(tokenize_function, batched=True).remove_columns(["model_input"])
tokenized_test.set_format("torch")

# 3. 执行快速推理
print("🚀 启动模型推理，请耐心等待进度条跑完...")
trainer = Trainer(model=model, args=TrainingArguments(output_dir="./tmp", per_device_eval_batch_size=32, report_to="none"))
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=-1)

# 把预测结果存入 df_test，供下一个单元格使用
df_test['predicted_label'] = y_pred
print("\n✅ 推理彻底完成！你可以安全地运行下一个单元格了。")

正在加载模型与测试集...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

正在处理文本张量化...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

🚀 启动模型推理，请耐心等待进度条跑完...



✅ 推理彻底完成！你可以安全地运行下一个单元格了。


In [9]:
import sys
# 确保能够导入上级目录的 utils
sys.path.append('..')
from utils.functions import calc_entropy, calc_relevance, calc_num_density

# 5. 核心过滤逻辑：找出越级误判样本
bad_cases_0_to_1 = df_test[(df_test['final_label'] == 0) & (df_test['predicted_label'] == 1)]
bad_cases_1_to_0 = df_test[(df_test['final_label'] == 1) & (df_test['predicted_label'] == 0)]
bad_cases_2_to_1 = df_test[(df_test['final_label'] == 2) & (df_test['predicted_label'] == 1)]
bad_cases_1_to_2 = df_test[(df_test['final_label'] == 1) & (df_test['predicted_label'] == 2)]

print("==================================================")
print("🧐 越级误判样本捕获统计报告")
print("==================================================")
print(f"✅ '0判1' (废话被抬高) : {len(bad_cases_0_to_1)} 条")
print(f"✅ '1判0' (中等被冤枉) : {len(bad_cases_1_to_0)} 条")
print(f"✅ '2判1' (高质量被降维) : {len(bad_cases_2_to_1)} 条")
print(f"✅ '1判2' (中等被高估) : {len(bad_cases_1_to_2)} 条\n")

# === 核心：定义统一的打印与特征追溯函数 ===
def print_bad_case_with_reason(df_cases, true_label_str, pred_label_str, case_name):
    if len(df_cases) == 0:
        return
        
    print(f"\n🧐 典型 Bad Case 抽样展示 ({case_name})")
    for idx, row in df_cases.sample(min(3, len(df_cases))).iterrows():
        print(f"【输入文本】:\n{row['model_input']}\n")
        print(f"【人工/规则标签】: {true_label_str}")
        print(f"【大模型预测标签】: {pred_label_str}")
        
        # 动态反算判案依据
        try:
            # 兼容 "[SEP]" 分隔格式 (你在 main.py 或特征提取时的拼接方式)
            if '[SEP]' in row['model_input']:
                parts = row['model_input'].split('[SEP]')
            else:
                # 兜底：如果是 "问：xxx 答：xxx" 格式
                parts = row['model_input'].replace('问：', '').split(' 答：')
                
            if len(parts) == 2:
                q_text, a_text = parts[0].strip(), parts[1].strip()
                
                # 重新计算原始特征
                entropy_val = calc_entropy(a_text)
                relevance_val = calc_relevance(q_text, a_text)
                num_density_val = calc_num_density(a_text)
                len_val = len(a_text)

                print(f" ‣ 信息熵 (权重 42%): {entropy_val:.4f}  <-- 若此项过高，容易强行拉高总分")
                print(f" ‣ 相关性 (权重 23%): {relevance_val:.4f}  <-- 若此项过低，说明答非所问")
                print(f" ‣ 数字密度 (权重 15%): {num_density_val:.4f}")
                print(f" ‣ 回答长度 (惩罚 -6%): {len_val}")
            else:
                print("\n特征追溯失败：未能成功拆分问答上下文。")
        except Exception as e:
            print(f"\n特征追溯失败：{e}")
            
        print("-" * 70)

# === 直接调用函数执行打印 ===
print_bad_case_with_reason(bad_cases_0_to_1, "0 (低质量/答非所问)", "1 (中等质量)", "True: 0 -> Pred: 1")
print_bad_case_with_reason(bad_cases_1_to_0, "1 (中等质量)", "0 (低质量/答非所问)", "True: 1 -> Pred: 0")
print_bad_case_with_reason(bad_cases_2_to_1, "2 (高质量)", "1 (中等质量)", "True: 2 -> Pred: 1")
print_bad_case_with_reason(bad_cases_1_to_2, "1 (中等质量)", "2 (高质量)", "True: 1 -> Pred: 2")

🧐 越级误判样本捕获统计报告
✅ '0判1' (废话被抬高) : 27 条
✅ '1判0' (中等被冤枉) : 34 条
✅ '2判1' (高质量被降维) : 65 条
✅ '1判2' (中等被高估) : 31 条


🧐 典型 Bad Case 抽样展示 (True: 0 -> Pred: 1)
【输入文本】:
请问，2016年及以后，公司年报上均不再披露自产水泥数量了吗？可否告知2016至2020年，各年间公司生产水泥的吨数？[SEP]感谢您对海螺水泥的关注！公司在历年年报中均有披露水泥和熟料销量数据。

【人工/规则标签】: 0 (低质量/答非所问)
【大模型预测标签】: 1 (中等质量)
 ‣ 信息熵 (权重 42%): 3.1219  <-- 若此项过高，容易强行拉高总分
 ‣ 相关性 (权重 23%): 0.0857  <-- 若此项过低，说明答非所问
 ‣ 数字密度 (权重 15%): 0.0000
 ‣ 回答长度 (惩罚 -6%): 34
----------------------------------------------------------------------
【输入文本】:
谢谢你们及时回答了我的问题。这次我想问问，到目前钾肥具有异议股回购申请权的股票数量还有多少？如果现在的不知道，那么开始的数量有多少？谢谢。[SEP]开始的数量请参阅相关股东大会决议公告，约600多万股。

【人工/规则标签】: 0 (低质量/答非所问)
【大模型预测标签】: 1 (中等质量)
 ‣ 信息熵 (权重 42%): 2.8074  <-- 若此项过高，容易强行拉高总分
 ‣ 相关性 (权重 23%): 0.1176  <-- 若此项过低，说明答非所问
 ‣ 数字密度 (权重 15%): 0.0357
 ‣ 回答长度 (惩罚 -6%): 27
----------------------------------------------------------------------
【输入文本】:
目前众多国内汽车企业都已经出了2021年1-2汽车销量同比大增经营快报，汽车协会也发布了1-2月份汽车销量大增的报告，请问公司2021年1-2销量的销售数据如何，公司销售的汽车端前装系统，无重大变化吗？如果真没变化公司可要凉凉了，腾讯怪不得这么减持呢？[SE